# 01 - EDA AND CLEANING

Phone price dataset: load, inspect, clean, explore and visualize.
Output: `data/processed/mobile_phone_price_clean.csv` (read by notebook 02).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import math
import os

os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

print("libraries imported successfully")

# LOAD THE DATA INTO A PANDAS DATAFRAME

In [ ]:
df = pd.read_csv("../data/raw/mobile_phone_price.csv")

df

# INSPECT THE DATA

In [ ]:
print("\nshape of the dataset",df.shape)
print('\nFeature Names')
print(df.columns.tolist())
print('\nBasic Information')
print(df.info())

# Data Formatting

In [ ]:
df.columns=df.columns.str.strip()
df['Storage']=df['Storage'].str.replace('GB','').astype(int)
df['RAM']=df['RAM'].str.replace('GB','').astype(int)
df['Price ($)']=df['Price ($)'].str.replace('$','').str.replace(',','').astype(int)
df['Screen Size (inches)'] = df['Screen Size (inches)'].str.extract(r'(\d+\.?\d*)')[0].astype(float)

def sum_camera_mp(val):
    total = 0.0
    for part in str(val).split('+'):
        clean_num = ''.join(c for c in part if c.isdigit() or c == '.')
        if clean_num:
            total += float(clean_num)
    return total

df['Camera (MP)'] = df['Camera (MP)'].apply(sum_camera_mp)

# DESCRIBE THE DATA

In [ ]:
display(df.describe().round(2))
print('\nNumber of Unique Brands: ',df['Brand'].nunique())
print('Number of Unique Models: ',df['Model'].nunique())

# CHECK FOR MISSING DATA

In [ ]:
print("\nMissing values:")
display(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

# CHECK FOR DUPLICATE DATA

In [ ]:
duplicates=df.duplicated().sum()
print('Number of duplicate rows: ',duplicates)

if duplicates>0:
    df=df.drop_duplicates()
    print('\nDuplicates removed')
else:
    print('No duplicates')

# BASIC DATA CLEANING

In [ ]:
df.columns = [
    "Brand",
    "Model",
    "Storage",
    "RAM",
    "Screen_size",
    "Camera",
    "Battery_cap",
    "Price"
]

print("Cleaned column names:")
print(df.columns.tolist())

# EXPLORE THE DATA

In [ ]:
print("Number of observations per Brand:")
display(df["Brand"].value_counts())

print("\nAverage measurements by Brand:")
display(
    df.groupby("Brand")[
        ["Storage",
         "RAM",
         "Screen_size",
         "Camera",
         "Battery_cap",
         "Price"]
    ].mean().round(2)
)

# VISUALIZE THE DATA

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})

fig,axes=plt.subplots(1,3 , figsize=(14,4))
fig.suptitle('Target variable: Price' , fontsize='20',fontweight='bold')

# Histogram
axes[0].hist(df['Price'], bins=20, color='lightblue', edgecolor='white', lw=0.5)
axes[0].axvline(df['Price'].mean(), color='red', ls='--',lw=2, label='Mean')
axes[0].axvline(df['Price'].median(), color='green', ls='--', lw=2, label='Median')
axes[0].set_xlabel("Price")
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=10)
 
# KDE
sns.kdeplot(df['Price'], ax=axes[1],fill=True, color='brown')
axes[1].axvline(df['Price'].mean(), color='red', ls='--',lw=2, label='Mean')
axes[1].axvline(df['Price'].median(), color='green', ls='--', lw=2, label='Median')
axes[1].set_xlabel('Price')
axes[1].legend(fontsize=10)

# Box Plot
axes[2].boxplot(df['Price'],patch_artist=True, boxprops=dict(facecolor='lightblue',alpha=0.8))
axes[2].set_ylabel('Price')
axes[2].set_xticks([])

plt.tight_layout()
plt.savefig("../reports/figures/target_variable_price.png", bbox_inches="tight", dpi=110)
plt.show()

In [ ]:
fig, axes=plt.subplots(3,2, figsize=(12,12))
fig.suptitle('Features vs Price', fontsize=13, fontweight='bold')

feats=['Storage', 'RAM', 'Screen_size', 'Camera', 'Battery_cap']
labels=["Storage (GB)","RAM (GB)","Screen Size (inches)","Camera (MP)","Battery Capacity (mAh)"]

for ax,feat,label in zip(axes.flatten(),feats,labels):
    ax.scatter(df[feat],df['Price'], alpha=0.4, s=15, color='#4C72B0')
    ax.set_title(f'{feat} vs Price')
    ax.set_xlabel(label)
    ax.set_ylabel('Price')

brand_avg_price=df.groupby("Brand")['Price'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=brand_avg_price, x='Brand', y='Price', ax=axes[2][1])
plt.title('Average Phone Price by Brand')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("../reports/figures/features_vs_price.png", bbox_inches="tight", dpi=110)
plt.show()

# CORRELATION ANALYSIS

In [ ]:
fig, ax=plt.subplots(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True),annot=True, cmap='coolwarm',fmt='.2f')
ax.set_title("Feature Correlation Matrix")
plt.savefig("../reports/figures/correlation_matrix.png", bbox_inches="tight", dpi=110)
plt.show() 

# SAVE THE CLEANED DATA

In [ ]:
df.to_csv("../data/processed/mobile_phone_price_clean.csv", index=False)
print("Saved cleaned data:", df.shape)